# 08 — Observability & Reliability

## Overview

This lab builds an **operational health-check framework** for the entire Fabric Lakehouse pipeline stack. Instead of discovering failures through user complaints or missing Power BI data, this notebook proactively evaluates three SLA dimensions — freshness, error rate, and latency — and raises structured alerts to a queryable Delta table (and optionally to Microsoft Teams).**Note:** Set `p_teams_webhook_url` only if you have a real Teams webhook; leave blank to skip notification.

**Prerequisites:** Scenarios 01–05 have been run so `silver.orders` exists. The `pipeline_run_log` and `pipeline_alerts` tables are auto-created on first run.  

**What you will learn**

- How to create and use a `pipeline_run_log` table as a centralised audit trail for all pipeline executions- How to send structured alert payloads to a Microsoft Teams channel via an incoming webhook

- How to define threshold-based SLA checks and route violations to `pipeline_alerts` with severity levels (`WARN` / `CRITICAL`)- How to measure **run latency** from the last N successful executions and alert if the average exceeds an SLA budget

- How to detect **data staleness** by measuring the age of the latest Silver timestamp against a configurable freshness window- How to compute **error rates** over a rolling 24-hour window from the run log

### Cell 1 — SLA threshold parameters
Four configurable thresholds define what "healthy" means for this pipeline stack:
- **`p_max_freshness_hours`** — the maximum acceptable age (in hours) of the latest row in `silver.orders`. A value of 25 comfortably covers a daily batch pipeline with a 1-hour processing window while still alerting if a full day was missed.
- **`p_max_error_rate_pct`** — the percentage of pipeline runs (in the last 24 hours) that may fail before triggering a warning. 10% means up to 1 in 10 runs may fail without alert — tighten to 0% for zero-tolerance production pipelines.
- **`p_max_latency_s`** — the maximum acceptable average duration (seconds) of the last 5 successful runs. 600 s (10 minutes) is a reasonable baseline for a notebook-per-layer architecture; adjust based on your actual measured baselines.
- **`p_teams_webhook_url`** — leave blank during lab exercises; populate with a real Teams incoming webhook URL in integration environments.

### Cell 2 — Configuration & check-run identity
`CHECK_RUN_ID` is a UUID generated fresh each time this notebook runs. Every alert written to `pipeline_alerts` in this session shares the same run ID, making it trivial to query "all alerts from this specific health-check invocation" without ambiguity. The `alerts` list accumulates all triggered alert objects in memory so they can be batch-sent to Teams at the end without hitting the webhook repeatedly per check.

### Cell 3 — Bootstrap control tables
`CREATE TABLE IF NOT EXISTS` is idempotent — running this cell multiple times never overwrites existing data. `pipeline_run_log` is the audit trail where each pipeline notebook should append one row per execution (start time, end time, rows read/written, status). `pipeline_alerts` is the alert inbox: every check violation writes a structured row with severity, metric value, threshold, and status (`OPEN` / `RESOLVED`). Storing alerts in Delta instead of plain logs makes them queryable from Power BI or the Fabric SQL endpoint, enabling real-time operational dashboards.

### Cell 4 — `raise_alert` helper
Encapsulates the alert-writing logic so each check below is a single call rather than repeated boilerplate. Each call: (1) assigns a unique `alert_id` UUID so individual alerts can be referenced and resolved independently; (2) appends a structured row to `pipeline_alerts`; (3) appends the same dict to the in-memory `alerts` list used by the Teams notification cell at the end. The `severity` parameter (`"WARN"` / `"CRITICAL"`) allows downstream consumers (dashboards, on-call tools) to route differently based on urgency.

### Cell 5 — Check 1: Data freshness
Queries `MAX(_silver_ts)` from `silver.orders` — the most recent row written by Scenario 02's MERGE. Computes the age of that timestamp relative to `datetime.now(UTC)` in hours. If the lag exceeds `p_max_freshness_hours`, a `WARN` alert is raised. If the table is empty (no Silver data at all), a `CRITICAL` alert is raised instead. A try/except wraps the entire check so a missing table or Spark error does not crash the notebook — it raises a `CRITICAL` alert and continues evaluating the remaining checks.

### Cell 6 — Check 2: Error rate (rolling 24 hours)
Queries `pipeline_run_log` for all runs completed in the last 24 hours and computes the ratio of `FAILED` to total runs as a percentage. This measures the **stability** of the pipeline stack over a recent window — a single isolated failure is often acceptable, but a sustained 30% failure rate signals a systemic problem (broken source connection, schema drift, expired credential). The 24-hour window is intentionally short so intermittent issues are detected quickly rather than being diluted by a week of successful historical runs.

### Cell 7 — Check 3: Run latency (last 5 successful runs)
Averages `duration_s` from the 5 most recent `SUCCEEDED` entries in `pipeline_run_log`. Latency creep is a common failure mode: a pipeline that ran in 3 minutes when the table had 1M rows may take 45 minutes once it has grown to 100M rows without schema or code changes. Monitoring the trailing average of recent successful runs catches this gradual degradation before it breaches SLA windows. The `LIMIT 5` keeps the sample small and recent — older runs from before a tuning exercise should not skew the current baseline.

### Cell 8 — Optional Teams notification
If `p_teams_webhook_url` is populated and at least one alert was raised, this cell sends a **Microsoft Teams MessageCard** via HTTP POST to the webhook URL. The payload groups all alerts from this run into a single card with a facts table — one row per alert (check name + message) — so the on-call engineer receives a clear summary rather than one notification per check. `urllib.request` is used (stdlib, no extra install needed) with a 10-second timeout to prevent the notebook from hanging indefinitely on a slow or unreachable webhook endpoint. Never log the webhook URL to output — treat it as a secret.

In [ ]:
p_max_freshness_hours = 25    # alert if Silver not updated within this window
p_max_error_rate_pct  = 10.0  # alert if >X% of last-24h runs failed
p_max_latency_s       = 600   # alert if avg run duration exceeds this
p_teams_webhook_url   = ""    # optional — leave blank to skip Teams notification

In [ ]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
from uuid import uuid4
import json

LH          = "lh_advanced_scenarios"
RUN_LOG     = f"{LH}.bronze.pipeline_run_log"
ALERT_TABLE = f"{LH}.bronze.pipeline_alerts"

CHECK_RUN_ID = str(uuid4())
alerts = []   # collect check results

In [ ]:
# ── Bootstrap tables if absent ────────────────────────────────────────────────
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {RUN_LOG} (
        run_id          STRING     NOT NULL,
        pipeline_name   STRING     NOT NULL,
        entity_name     STRING,
        status          STRING     NOT NULL,
        rows_read       BIGINT,
        rows_written    BIGINT,
        rows_quarantine BIGINT,
        error_message   STRING,
        started_at      TIMESTAMP,
        ended_at        TIMESTAMP,
        duration_s      DOUBLE
    ) USING DELTA
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {ALERT_TABLE} (
        alert_id        STRING     NOT NULL,
        check_name      STRING     NOT NULL,
        severity        STRING     NOT NULL,
        message         STRING,
        metric_value    DOUBLE,
        threshold       DOUBLE,
        status          STRING     NOT NULL,
        raised_at       TIMESTAMP,
        resolved_at     TIMESTAMP
    ) USING DELTA
""")
print("Control tables ready.")

In [ ]:
# ── Helper: raise alert ───────────────────────────────────────────────────────
def raise_alert(check_name: str, severity: str, message: str, metric: float, threshold: float):
    row = {
        "alert_id":     str(uuid4()),
        "check_name":   check_name,
        "severity":     severity,
        "message":      message,
        "metric_value": metric,
        "threshold":    threshold,
        "status":       "OPEN",
        "raised_at":    datetime.now(timezone.utc),
        "resolved_at":  None,
    }
    spark.createDataFrame([row]).write.format("delta").mode("append").saveAsTable(ALERT_TABLE)
    alerts.append(row)
    print(f"ALERT [{severity}] {check_name}: {message}")

In [ ]:
# ── Check 1: Data freshness ───────────────────────────────────────────────────
try:
    last_silver = spark.sql(
        f"SELECT MAX(_silver_ts) AS ts FROM {LH}.silver.orders"
    ).collect()[0]["ts"]

    if last_silver is None:
        raise_alert("data_freshness", "CRITICAL", "silver.orders has no data", 0, p_max_freshness_hours)
    else:
        lag_hours = (datetime.now(timezone.utc) - last_silver.replace(tzinfo=timezone.utc)).total_seconds() / 3600
        print(f"Freshness lag: {lag_hours:.1f} h (threshold: {p_max_freshness_hours} h)")
        if lag_hours > p_max_freshness_hours:
            raise_alert("data_freshness", "WARN",
                        f"silver.orders last updated {lag_hours:.1f}h ago",
                        lag_hours, p_max_freshness_hours)
        else:
            print("  ✓ Freshness OK")
except Exception as e:
    raise_alert("data_freshness", "CRITICAL", str(e), -1, p_max_freshness_hours)

In [ ]:
# ── Check 2: Error rate in last 24 h ─────────────────────────────────────────
try:
    rate_row = spark.sql(f"""
        SELECT
            COUNT(*) AS total,
            SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) AS failures
        FROM {RUN_LOG}
        WHERE ended_at >= CURRENT_TIMESTAMP() - INTERVAL 24 HOURS
    """).collect()[0]

    total    = rate_row["total"] or 0
    failures = rate_row["failures"] or 0
    rate     = (failures / total * 100) if total > 0 else 0.0
    print(f"Error rate (24h): {rate:.1f}% ({failures}/{total} runs)")
    if rate > p_max_error_rate_pct:
        raise_alert("error_rate_24h", "WARN",
                    f"{rate:.1f}% failure rate in last 24h ({failures}/{total} runs)",
                    rate, p_max_error_rate_pct)
    else:
        print("  ✓ Error rate OK")
except Exception as e:
    raise_alert("error_rate_24h", "CRITICAL", str(e), -1, p_max_error_rate_pct)

In [ ]:
# ── Check 3: Average latency of last 5 successful runs ────────────────────────
try:
    latency = spark.sql(f"""
        SELECT AVG(duration_s) AS avg_s
        FROM (
            SELECT duration_s FROM {RUN_LOG}
            WHERE status = 'SUCCEEDED'
            ORDER BY ended_at DESC
            LIMIT 5
        )
    """).collect()[0]["avg_s"] or 0.0

    print(f"Avg latency (last 5 runs): {latency:.1f}s (threshold: {p_max_latency_s}s)")
    if latency > p_max_latency_s:
        raise_alert("run_latency", "WARN",
                    f"Average run duration {latency:.1f}s exceeds SLA of {p_max_latency_s}s",
                    latency, float(p_max_latency_s))
    else:
        print("  ✓ Latency OK")
except Exception as e:
    raise_alert("run_latency", "CRITICAL", str(e), -1, float(p_max_latency_s))

In [ ]:
# ── Optional: Teams notification ─────────────────────────────────────────────
if p_teams_webhook_url and alerts:
    import urllib.request
    payload = {
        "@type": "MessageCard",
        "@context": "http://schema.org/extensions",
        "themeColor": "FF0000",
        "summary": f"{len(alerts)} Fabric Lakehouse alert(s) raised",
        "sections": [{
            "activityTitle": f"{len(alerts)} alert(s) — {datetime.now(timezone.utc).isoformat()}",
            "facts": [{"name": a["check_name"], "value": a["message"]} for a in alerts],
        }]
    }
    req = urllib.request.Request(
        p_teams_webhook_url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    with urllib.request.urlopen(req, timeout=10) as resp:
        print(f"Teams notification sent: HTTP {resp.status}")

print(f"Observability checks complete. Alerts raised: {len(alerts)}")